# 🧬 De Novo Immunopeptidomics — CNN-LSTM Training

**Target:** Exact peptide accuracy ≥ 30%  
**GPU required:** Enable via *Runtime → Change runtime type → T4 GPU*

---
### Before you start
This notebook is optimized for **large datasets**. You will need to upload your data archive (`colab_data.tar.gz`) to Google Drive first.

1. **Upload `colab_data.tar.gz`** from your computer to the root of your Google Drive.
2. Click **Runtime → Change runtime type** and select **T4 GPU**.
3. Run **Section 1** to verify you have a GPU.
4. Run **Section 2** to clone the repository.
5. Run **Section 3** to mount Drive and extract your data to the fast Colab SSD.
6. Run **Section 4** to install dependencies.
7. Run **Section 5** to start training.
8. Run **Section 6** to save checkpoints back to your Google Drive.

## Section 1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Section 2 — Clone the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/chandan-rawallab/immunodenovorepo.git'
REPO_DIR = '/content/immunodenovorepo'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## Section 3 — Mount Google Drive & Extract Data

This step will mount your Google Drive and copy the compressed `colab_data.tar.gz` to Colab's fast local storage before extracting it. This avoids slow network reading during training.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Setup local Colab directories
DATA_DIR = '/content/immunodenovorepo/data'
MGF_DIR = DATA_DIR + '/mgf'
RESULTS = '/content/immunodenovorepo/results'
os.makedirs(MGF_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

ARCHIVE_PATH = '/content/drive/MyDrive/colab_data.tar.gz'

if not os.path.exists(ARCHIVE_PATH):
    raise FileNotFoundError("⚠️  Cannot find colab_data.tar.gz in the root of your Google Drive. Please upload it first.")

# 3. Extract directly to local Colab SSD for maximum speed
print("Extracting data... this may take a few minutes.")
!tar -xzf {ARCHIVE_PATH} -C {MGF_DIR} --wildcards '*.mgf' 2>/dev/null || true
!tar -xzf {ARCHIVE_PATH} -C {RESULTS} --wildcards '*.tsv' 2>/dev/null || true
!tar -xzf {ARCHIVE_PATH} -C {MGF_DIR} 2>/dev/null || true # fallback
!mv {MGF_DIR}/*.tsv {RESULTS}/ 2>/dev/null || true # cleanup

print("✅ Data extracted to local storage.")
!ls -lh {RESULTS}/immunopeptidome_psms.tsv
!echo "Number of MGF files extracted:" && ls -1 {MGF_DIR}/*.mgf | wc -l

## Section 4 — Install Dependencies

In [ ]:
%%bash
pip install -q pyteomics lxml pandas scikit-learn tqdm
echo '✅ Dependencies installed.'

## Section 5 — Run Training

Instead of importing the script, we run it directly via the command-line interface. This avoids any Python import syntax issues with filenames beginning with numbers.

In [ ]:
# Run training script via command line
!python3 src/training/05_train_denovo_model.py \
    --mgf-dir /content/immunodenovorepo/data/mgf \
    --psm-file /content/immunodenovorepo/results/immunopeptidome_psms.tsv \
    --checkpoint-dir /content/immunodenovorepo/results/checkpoints_v3 \
    --batch-size 32 \
    --epochs 150 \
    --lr 3e-4 \
    --num-workers 2 \
    --t0 10 \
    --t-mult 2 \
    --label-smoothing 0.10 \
    --patience 25 \
    --grad-accum 2

## Section 6 — Save Checkpoints to Google Drive

Run this immediately after training finishes to ensure you don't lose the checkpoints when the Colab instance disconnects.

In [ ]:
import glob
import shutil
import os

CHECKPOINT_DIR = '/content/immunodenovorepo/results/checkpoints_v3'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/immunodenovo_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

checkpoints = glob.glob(CHECKPOINT_DIR + '/*')
if not checkpoints:
    print('⚠️ No checkpoints found — did training complete?')
else:
    for path in checkpoints:
        fname = os.path.basename(path)
        dest = os.path.join(DRIVE_CKPT_DIR, fname)
        print(f'Copying {fname} to Google Drive...')
        shutil.copy2(path, dest)
    print(f'\n✅ Checkpoints successfully saved to: {DRIVE_CKPT_DIR}')